# DL_regression_diabetes_

내장 diabetes 데이터의 1년 후 질병 진행도를 예측합니다.

※ 회귀 정답은 과제 기준 확인 전의 예제 설정입니다.


## 1. 라이브러리와 재현성
고정 시드를 사용하지만 실행 장치와 라이브러리 버전에 따라 결과가 달라질 수 있습니다.


In [ ]:
import os, sys, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay,
    mean_absolute_error, mean_squared_error, r2_score)

SEED = 42
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
EPOCHS = 50
BATCH_SIZE = 32
print('Python:', sys.version.split()[0], 'TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 2. 데이터 불러오기


In [ ]:
from sklearn.datasets import load_diabetes
USE_UPLOAD = False
TARGET_COLUMN = 'target'  # 실제 CSV의 연속 정답 컬럼 이름으로 수정
if USE_UPLOAD:
    from google.colab import files
    import io
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('CSV 하나만 업로드하세요.')
    df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
else:
    # 원시 입력을 불러온 뒤 훈련 분할에서만 표준화
    dataset = load_diabetes(as_frame=True, scaled=False)
    df = dataset.frame.copy()
    TARGET_COLUMN = 'target'
print('Shape:', df.shape)
print(df.head())


## 3. 입력 X와 정답 y


In [ ]:
assert TARGET_COLUMN in df.columns
df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
X = df.drop(columns=[TARGET_COLUMN])
y = pd.to_numeric(df[TARGET_COLUMN], errors='raise').to_numpy(dtype='float32')
assert np.isfinite(y).all() and len(np.unique(y)) > 2, '연속형 정답이 필요합니다.'
STRATIFY = False
plt.hist(y, bins=25); plt.xlabel('Disease progression (example)'); plt.ylabel('Count'); plt.show()


## 4. 데이터 분할 및 전처리
전체의 약 64%/16%/20%를 훈련/검증/테스트로 사용합니다. 테스트는 최종 평가에만 사용합니다.


In [ ]:
# 먼저 분할한 뒤, 훈련 데이터에서만 대치값/표준화 기준을 학습합니다.
idx = np.arange(len(y))
trainval_idx, test_idx = train_test_split(idx, test_size=0.2,
    random_state=SEED, stratify=y if STRATIFY else None)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.2,
    random_state=SEED, stratify=y[trainval_idx] if STRATIFY else None)
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]
print('Train / validation / test:', len(train_idx), len(val_idx), len(test_idx))
assert not set(train_idx) & set(test_idx)
assert not set(val_idx) & set(test_idx)


In [ ]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.columns.difference(num_cols).tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), num_cols),
    ('category', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])
X_train = preprocessor.fit_transform(X.iloc[train_idx]).astype('float32')
X_val = preprocessor.transform(X.iloc[val_idx]).astype('float32')
X_test = preprocessor.transform(X.iloc[test_idx]).astype('float32')
assert np.isfinite(X_train).all() and np.isfinite(X_test).all()
print('Encoded input shape:', X_train.shape)


## 5. 회귀 모델
은닉층 ReLU, 출력층 선형 뉴런 1개, 손실 MSE입니다. 정답도 훈련 데이터 기준으로 표준화하여 학습하고 평가할 때 원래 단위로 복원합니다.


In [ ]:
target_scaler = StandardScaler()
y_train_fit = target_scaler.fit_transform(y_train.reshape(-1, 1)).astype('float32')
y_val_fit = target_scaler.transform(y_val.reshape(-1, 1)).astype('float32')
METRIC = 'mae'


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:]),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='mse', metrics=[METRIC])
model.summary()


## 6. 학습
`fit()`에서 순전파, 손실 계산, 역전파, 가중치 갱신을 반복합니다. 검증 데이터로 조기 종료 시점을 선택합니다.


In [ ]:
# 검증 손실이 개선되지 않으면 중단하고 최적 가중치를 복원합니다.
callbacks = [tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True)]
history = model.fit(X_train, y_train_fit,
    validation_data=(X_val, y_val_fit), epochs=EPOCHS,
    batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2)


## 7. 학습 곡선
훈련 손실은 감소하지만 검증 손실이 증가하면 과적합을 의심합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key, ax in zip(['loss', METRIC], axes):
    ax.plot(history.history[key], label='Train')
    ax.plot(history.history['val_' + key], label='Validation')
    ax.set(xlabel='Epoch (zero-based)', ylabel=key, title=key)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 8. 독립 테스트 평가


In [ ]:
prediction_scaled = model.predict(X_test, verbose=0)
y_pred = target_scaler.inverse_transform(prediction_scaled).ravel()
# 아무 특성도 사용하지 않는 훈련 정답 평균 예측을 기준선으로 비교
baseline = np.full(len(y_test), y_train.mean())
def regression_scores(actual, predicted):
    return {'MAE': float(mean_absolute_error(actual, predicted)),
            'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
            'R2': float(r2_score(actual, predicted))}
metrics = regression_scores(y_test, y_pred)
print(pd.DataFrame({'DNN': metrics,
      'Train-mean baseline': regression_scores(y_test, baseline)}).T)
print('MAE/RMSE는 정답의 원래 단위이며 작을수록 좋습니다. R2는 음수일 수도 있습니다.')
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, y_pred, s=14, alpha=0.6)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], 'r--')
axes[0].set(xlabel='Actual', ylabel='Predicted', title='Actual vs predicted')
axes[1].scatter(y_pred, y_test - y_pred, s=14, alpha=0.6)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set(xlabel='Predicted', ylabel='Actual - predicted', title='Residuals')
plt.tight_layout(); plt.show()
print(pd.DataFrame({'actual': y_test[:10], 'predicted': y_pred[:10]}))


## 9. 결과 저장
Colab 파일 탭에서 모델·지표·예측 CSV를 다운로드할 수 있습니다. 모델 재사용 시 전처리 파일도 함께 보관하세요.


In [ ]:
import joblib
RESULT_DIR = Path('DL_regression_diabetes__results')
RESULT_DIR.mkdir(exist_ok=True)
model.save(RESULT_DIR / 'model.keras')
(RESULT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
pd.DataFrame({'actual': y_test, 'predicted': y_pred}).to_csv(RESULT_DIR / 'predictions.csv', index=False)
metadata = {'task': 'regression', 'seed': SEED, 'tensorflow': tf.__version__}
if 'preprocessor' in globals(): joblib.dump(preprocessor, RESULT_DIR / 'preprocessor.joblib')
if 'target_scaler' in globals(): joblib.dump(target_scaler, RESULT_DIR / 'target_scaler.joblib')
if 'class_names' in globals(): metadata['class_names'] = [str(x) for x in class_names]
(RESULT_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', RESULT_DIR)


## 참고 자료
- [scikit-learn load_diabetes: 정답 정의](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html)
